In [221]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import precision_score,  recall_score, f1_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


### Problem overview

The goal is to build a model that can detect whether a patient suffers from hypertrophic cardiomyopathy (`Cardiomegaly`) based on selected features such as measurements of heart and lung size, ratios and proportions describing the heart’s shape and contour, and geometric indicators capturing the structure and orientation of the heart.

Each row in the dataset represents one patient, and the target column — `Cardiomegaly` — indicates the diagnosis outcome:

1 → the patient suffers from Cardiomegaly

0 → the patient does not suffer from Cardiomegaly

By analyzing these features and training machine learning models, we aim to uncover patterns associated with the presence of cardiomegaly and evaluate how accurately our model can predict the condition in unseen patient data.


### Data Loading and Scaling

The first step is to load the dataset from the provided .csv file and select the relevant numerical features for modeling. Once the features are extracted, the dataset is split into a training set (80%) and a testing set (20%) to evaluate model performance on unseen data. Before training, it is essential to standardize the features so that all variables are on a comparable scale.

In [222]:
features = [
    "Heart width",
    "Lung width",
    "CTR - Cardiothoracic Ratio",
    "normalized_diff",
    "Inscribed circle radius",
    "Polygon Area Ratio",
    "Heart perimeter",
    # Space in task_data.csv "Heart area "
    "Heart area ",
    "Lung area"
]
target = "Cardiomegaly"

df = pd.read_csv("task_data.csv")

# Unification - swapping 0,0 to 0.0
for col in df.columns:
    if col not in ["ID", "Cardiomegaly"]:
        df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_scaled_train = scaler.fit_transform(X_train)
X_scaled_test = scaler.transform(X_test)

### Model Selection for Cardiomegaly Prediction

In order to achieve the most valid prediction, we need to choose models carefully considering that our database has only 37 records, each represented by 9 numerical features. Given the small dataset and numerical nature of the features, some models are more suitable than others.

#### 1. Support Vector Machine (SVM)

SVM is one of the best choices for this dataset because:

- It works well with **small to medium-sized datasets** such as ours.
- It is effective in **high-dimensional spaces**, which is useful even with only 9 features.
- By using kernel functions (e.g., RBF), it can handle **nonlinear relationships** between features and the target variable (`Cardiomegaly`).

#### 2. Logistic Regression (LR)

Logistic Regression is another  suitable model:

- Works well when the relationship between features and the target is **approximately linear**.
- It includes regularization (L1/L2) that helps prevent overfitting on small datasets.

#### 3.Models Rejected and Why

- Decision Tree (single tree)
  - It is **prone to overfitting** on very small datasets. Adjusting ('max_depth') and ('min_samples_split') might help, but results may still be unstable.

- k-Nearest Neighbors (k-NN)
  - Sensitive to **feature scaling** and noise. Small dataset and nonlinear relationships may lead to unstable predictions.

- Random Forest
  - Combines multiple trees to reduce overfitting, but with **only 37 records**, effectiveness is limited





### Support Vector Machine (SVM)

Since SVMs are sensitive to feature scaling, we include data normalization to ensure that all numerical features contribute equally to the model. The RBF kernel is chosen because it can capture nonlinear decision boundaries, which is useful for modeling complex relationships between the features and cardiomegaly. The regularization parameter C is set to 2, as this value provides a balance between underfitting and overfitting: with a small dataset of 37 records, a very high C could overfit the training data, while cross-validation results showed that C=2 achieves the best mean score. The gamma parameter is set to 'scale', which adapts the kernel width to the variance of the data. The class_weight is set to None, meaning that all classes are treated equally without re-weighting.

In [231]:
pipe_svc = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="rbf",
        C=2,
        gamma="scale",
        class_weight=None
    ))
])

pipe_svc.fit(X_train, y_train)

cv_score = np.round(cross_val_score(pipe_svc, X_train, y_train), 2)

print("Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {cv_score.mean():.3f}")
print(f"Standard deviation of CV score: {cv_score.std():.3f}")

Scores of training data cross-validation (each fold):
0.83
0.83
0.83
0.83
0.8

Cross-validation mean score: 0.824
Standard deviation of CV score: 0.012


### Logistic Regression (LR)

For Logistic Regression, we also apply feature scaling using StandardScaler, as normalization improves stability of the model. With the regularization parameter C set to 5, it allows the model to better fit our small dataset of 37 samples without underfitting, as confirmed by cross-validation. The penalty is set to l1 to perform feature selection by driving some coefficients exactly to zero, which can also improve interpretability and reduce overfitting on small datasets. The liblinear solver is chosen because it is compatible with l1 regularization, max_iter is set to 1000 to ensure convergence even if the data contains collinearities. Class_weight is set to None, treating classes equally since the dataset is relatively balanced.

In [235]:
pipe_log = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=5,
        penalty="l1",
        solver="liblinear",
        max_iter=1000,
        class_weight=None
    ))
])

pipe_log.fit(X_train, y_train)

cv_score = np.round(cross_val_score(pipe_log, X_train, y_train), 2)

print("Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {cv_score.mean():.3f}")
print(f"Standard deviation of CV score: {cv_score.std():.3f}")

Scores of training data cross-validation (each fold):
0.83
0.83
0.67
0.83
0.8

Cross-validation mean score: 0.792
Standard deviation of CV score: 0.062


### Running the Model on the Test Dataset

In [236]:
y_pred_svc  = pipe_svc.predict(X_test)
y_pred_log  = pipe_log.predict(X_test)

acc_svc  = accuracy_score(y_test, y_pred_svc)
acc_log  = accuracy_score(y_test, y_pred_log)

print(f"Accuracy on test set:")
print(f"- Accuracy of SVC model on test dataset:                    {acc_svc:.4f}")
print(f"- Accuracy of Logistic Regression model on test dataset:    {acc_log:.4f}")

prec_svc = precision_score(y_test, y_pred_svc)
prec_log = precision_score(y_test, y_pred_log)

print(f"Precision on test set:")
print(f"- Precision of SVC model on test dataset:                   {prec_svc:.4f}")
print(f"- Precision of Logistic Regression model on test dataset:   {prec_log:.4f}")

rec_svc = recall_score(y_test, y_pred_svc)
rec_log = recall_score(y_test, y_pred_log)

print(f"Recall on test set:")
print(f"- Recall of SVC model on test dataset:                      {rec_svc:.4f}")
print(f"- Recall of Logistic Regression model on test dataset:      {rec_log:.4f}")

f1_svc = f1_score(y_test, y_pred_svc)
f1_log = f1_score(y_test, y_pred_log)

print(f"F1-score on test set:")
print(f"- F1-cs of SVC model on test dataset:                       {f1_svc:.4f}")
print(f"- f1 of Logistic Regression model on test dataset:          {f1_log:.4f}")

Accuracy on test set:
- Accuracy of SVC model on test dataset:                    0.7500
- Accuracy of Logistic Regression model on test dataset:    0.6250
Precision on test set:
- Precision of SVC model on test dataset:                   0.7500
- Precision of Logistic Regression model on test dataset:   0.7143
Recall on test set:
- Recall of SVC model on test dataset:                      1.0000
- Recall of Logistic Regression model on test dataset:      0.8333
F1-score on test set:
- F1-cs of SVC model on test dataset:                       0.8571
- f1 of Logistic Regression model on test dataset:          0.7692


### Model Evaluation Results

| Classifier | Accuracy (CV Mean) | Accuracy (Test) | Precision (Test) | Recall (Test) | F1-score (Test) |
|-------------|-------------------|-----------------|------------------|----------------|-----------------|
| **Support Vector Machine (SVM)** | **82.4%** | **75.0%** | **75.0%** | **100.0%** | **85.7%** |
| **Logistic Regression** | **79.2%** | **62.5%** | **71.4%** | **83.3%** | **76.9%** |


### Final remarks

As we can see, the final results differ from the initially anticipated ones. During the optimization process, we worked only with the standardized training dataset. It is natural that performance decreases when evaluating on unseen test data. This difference reflects how the model reacts to new, previously unseen data.

The goal of this task was to use classical machine learning methods to predict whether a patient suffers from Cardiomegaly. With only 37 records, the dataset provides limited opportunities for any ML method to fully learn the underlying patterns. Therefore, prediction accuracy might not be as high as desired.

On the test set, the SVC model achieved an accuracy of 0.75, while the Logistic Regression model reached 0.625. This suggests that SVC was better able to capture the patterns in the data, likely due to its ability to handle nonlinear relationships via the RBF kernel.

Considering other evaluation metrics, SVC achieved high recall (1.0), precision (0.75), and F1-score (0.857). This indicates that the model was able to correctly identify all positive Cardiomegaly cases while maintaining a strong balance between detecting true positives and avoiding false positives. Logistic Regression, with lower precision (0.714) and recall (0.833), was slightly less effective in both identifying and distinguishing true positive cases.

To sum up, the SVC model performed better on this small dataset, benefiting from its ability to capture nonlinear relationships via the RBF kernel. In a medical diagnosis context, where missing a positive case is highly undesirable, the high recall of SVC makes it the more suitable choice. These results are reasonable and highlight the challenges of training reliable models on limited data.